# PDF đề toán -> Markdown + Images -> TeX bằng MinerU

Notebook này chạy trên Google Colab để OCR các PDF đề toán, chuẩn hóa ảnh theo câu hỏi, tạo file Markdown, file TeX và `image_mapping.json`.

## 1. Kết nối Colab bằng VS Code extension

Chạy notebook này ngay trong VS Code:

1. Mở notebook.
2. Bấm `Select Kernel` ở góc trên phải.
3. Chọn `Colab`.
4. Chọn `Auto Connect` hoặc `New Colab Server`; ưu tiên server có GPU nếu có.
5. Đăng nhập Google và bấm `Allow` khi extension yêu cầu.
6. Upload PDF vào Colab server bằng `Upload to Colab`, hoặc mount Drive bằng `Colab: Mount Google Drive to Server...`.

Notebook sẽ tự tìm PDF trong `/content/input_pdfs`, `/content`, `/content/HSG tinh` và `/content/drive/MyDrive`.

Lưu ý: notebook mặc định ép MinerU dùng backend `pipeline` để tránh lỗi thiếu `vllm` của backend `hybrid-engine`.

In [6]:
import os, sys, subprocess

def sh(cmd):
    print('\n$ ' + cmd)
    subprocess.run(cmd, shell=True, check=True)

sh(sys.executable + ' -m pip install -q -U pip')
sh(sys.executable + ' -m pip install -q -U mineru[all] pymupdf pillow python-slugify pandas')



$ /usr/bin/python3 -m pip install -q -U pip

$ /usr/bin/python3 -m pip install -q -U mineru[all] pymupdf pillow python-slugify pandas


## 2. Cấu hình đường dẫn

Nếu dùng extension, hãy upload PDF vào Colab server bằng `Upload to Colab`. Notebook sẽ scan các thư mục phổ biến nên không cần dùng `files.upload()`.

In [7]:
from pathlib import Path

INPUT_DIR = Path('/content/input_pdfs')
RAW_DIR = Path('/content/outputs/mineru_raw')
FINAL_DIR = Path('/content/outputs/final')
MD_DIR = FINAL_DIR / 'markdown'
TEX_DIR = FINAL_DIR / 'tex'
IMG_DIR = FINAL_DIR / 'images'

PDF_FILES = [
    'Đề 01.pdf',
    'Đề 3 HSG.pdf',
    'Cau truc de thi HSG 10.pdf',
    '10-HSG-TRẤN BIÊN 2026-AZOTA.pdf',
]

PDF_SEARCH_DIRS = [
    INPUT_DIR,
    Path('/content'),
    Path('/content/HSG tinh'),
    Path('/content/drive/MyDrive'),
]

FORCE_PIPELINE_BACKEND = True

for folder in [INPUT_DIR, RAW_DIR, FINAL_DIR, MD_DIR, TEX_DIR, IMG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('PDF search dirs:')
for folder in PDF_SEARCH_DIRS:
    print('-', folder)
print('Expected PDFs:', PDF_FILES)
print('FORCE_PIPELINE_BACKEND =', FORCE_PIPELINE_BACKEND)

PDF search dirs:
- /content/input_pdfs
- /content
- /content/HSG tinh
- /content/drive/MyDrive
Expected PDFs: ['Đề 01.pdf', 'Đề 3 HSG.pdf', 'Cau truc de thi HSG 10.pdf', '10-HSG-TRẤN BIÊN 2026-AZOTA.pdf']
FORCE_PIPELINE_BACKEND = True


In [8]:
def find_pdf_file(pdf_name):
    for folder in PDF_SEARCH_DIRS:
        candidate = folder / pdf_name
        if candidate.exists():
            return candidate
    for folder in PDF_SEARCH_DIRS:
        if folder.exists():
            matches = list(folder.rglob(pdf_name))
            if matches:
                return matches[0]
    return None

found = {name: find_pdf_file(name) for name in PDF_FILES}
for name, path in found.items():
    print(('FOUND   ' if path else 'MISSING ') + name + (f' -> {path}' if path else ''))

missing = [name for name, path in found.items() if path is None]
if missing:
    print('\nUpload các file còn thiếu bằng VS Code Explorer > right click > Upload to Colab, rồi chạy lại cell này:')
    for name in missing:
        print('-', name)
else:
    print('\nAll PDFs are available on the Colab server.')

FOUND   Đề 01.pdf -> /content/Đề 01.pdf
FOUND   Đề 3 HSG.pdf -> /content/Đề 3 HSG.pdf
FOUND   Cau truc de thi HSG 10.pdf -> /content/Cau truc de thi HSG 10.pdf
FOUND   10-HSG-TRẤN BIÊN 2026-AZOTA.pdf -> /content/10-HSG-TRẤN BIÊN 2026-AZOTA.pdf

All PDFs are available on the Colab server.


## 3. Hàm xử lý MinerU, Markdown, ảnh và TeX

In [9]:
import json, re, shutil, subprocess
from collections import defaultdict
from urllib.parse import unquote
from slugify import slugify

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
QUESTION_RE = re.compile(r'(?:\*\*)?\s*C[âa]u\s*(\d+)', re.IGNORECASE)
MD_IMG_RE = re.compile(r'!\[[^\]]*\]\(([^)]+)\)')
HTML_IMG_RE = re.compile(r'<img[^>]+src=[\"\']([^\"\']+)[\"\'][^>]*>', re.IGNORECASE)
MATH_RE = re.compile(r'(\$\$.*?\$\$|\$.*?\$|\\\[.*?\\\]|\\\(.*?\\\))', re.DOTALL)

def doc_slug(path):
    return slugify(Path(path).stem, lowercase=True) or 'document'

def run_mineru(pdf_path):
    pdf_path = Path(pdf_path)
    out_dir = RAW_DIR / doc_slug(pdf_path)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['mineru', '-p', str(pdf_path), '-o', str(out_dir)]
    if FORCE_PIPELINE_BACKEND:
        cmd += ['-b', 'pipeline']
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-5000:])
        raise RuntimeError('MinerU failed for ' + pdf_path.name)
    return out_dir

def find_markdown(raw_doc_dir):
    candidates = list(Path(raw_doc_dir).rglob('*.md'))
    if not candidates:
        raise FileNotFoundError('No Markdown found in ' + str(raw_doc_dir))
    return sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)[0]

def locate_image(src, md_dir, raw_doc_dir):
    s = unquote(str(src)).strip().strip(chr(34)).strip(chr(39))
    if s.startswith(('http://', 'https://', 'data:')):
        return None
    s = s.split('#')[0].split('?')[0]
    for base in [md_dir, Path(raw_doc_dir)]:
        candidate = (base / s).resolve()
        if candidate.exists():
            return candidate
    name = Path(s).name
    matches = [p for p in Path(raw_doc_dir).rglob(name) if p.suffix.lower() in IMAGE_EXTS]
    return matches[0] if matches else None

def normalize_markdown_and_images(pdf_path, raw_doc_dir):
    pdf_path = Path(pdf_path)
    slug = doc_slug(pdf_path)
    md_path = find_markdown(raw_doc_dir)
    text = md_path.read_text(encoding='utf-8', errors='ignore')
    current_question = 'unknown'
    counts = defaultdict(int)
    mappings = []
    out_lines = []

    def replace_src(src):
        nonlocal current_question
        src_path = locate_image(src, md_path.parent, raw_doc_dir)
        if src_path is None:
            return src
        q = current_question or 'unknown'
        counts[q] += 1
        q_label = ('cau' + str(int(q)).zfill(2)) if str(q).isdigit() else 'unknown'
        new_name = f'{slug}_{q_label}_{counts[q]:02d}{src_path.suffix.lower()}'
        dest = IMG_DIR / new_name
        shutil.copy2(src_path, dest)
        mappings.append({
            'document': pdf_path.name,
            'document_slug': slug,
            'question': q,
            'image': new_name,
            'source_image': str(src_path),
            'markdown_path': '../images/' + new_name
        })
        return '../images/' + new_name

    for line in text.splitlines():
        match = QUESTION_RE.search(line)
        if match:
            current_question = match.group(1)
        line = MD_IMG_RE.sub(lambda m: '![image](' + replace_src(m.group(1)) + ')', line)
        line = HTML_IMG_RE.sub(lambda m: '![image](' + replace_src(m.group(1)) + ')', line)
        out_lines.append(line)

    out_md = MD_DIR / (slug + '.md')
    out_md.write_text('\n'.join(out_lines).strip() + '\n', encoding='utf-8')
    return out_md, mappings

def escape_tex_text(s):
    repl = {'&': r'\&', '%': r'\%', '#': r'\#', '_': r'\_', '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}'}
    return ''.join(repl.get(ch, ch) for ch in s)

def inline_md_to_tex(line):
    line = line.replace('**', '')
    parts = MATH_RE.split(line)
    out = []
    for part in parts:
        if not part:
            continue
        out.append(part if MATH_RE.fullmatch(part) else escape_tex_text(part))
    return ''.join(out)

def markdown_to_tex(md_path, title):
    lines = Path(md_path).read_text(encoding='utf-8').splitlines()
    body = []
    for line in lines:
        raw = line.strip()
        if not raw:
            body.append('')
            continue
        img = MD_IMG_RE.search(raw)
        if img:
            body += [r'\begin{center}', r'\includegraphics[width=0.82\linewidth]{' + img.group(1).replace('\\', '/') + '}', r'\end{center}']
            continue
        if raw.startswith('#'):
            level = len(raw) - len(raw.lstrip('#'))
            cmd = 'section*' if level <= 1 else 'subsection*'
            body.append('\\' + cmd + '{' + inline_md_to_tex(raw[level:].strip()) + '}')
            continue
        if QUESTION_RE.match(raw):
            body.append('\n' + r'\medskip')
            body.append(r'\noindent ' + inline_md_to_tex(raw) + r'\par')
        else:
            body.append(inline_md_to_tex(raw) + r'\par')

    preamble = r'''\documentclass[12pt,a4paper]{article}
\usepackage{fontspec}
\usepackage[vietnamese]{babel}
\usepackage{amsmath,amssymb,mathtools}
\usepackage{graphicx}
\usepackage{geometry}
\usepackage{enumitem}
\geometry{margin=2cm}
\setmainfont{TeX Gyre Termes}
\setlength{\parindent}{0pt}
\setlength{\parskip}{4pt}
\begin{document}
\begin{center}
{\Large\bfseries TITLE_PLACEHOLDER}
\end{center}
'''.replace('TITLE_PLACEHOLDER', escape_tex_text(title))
    tex = preamble + '\n'.join(body) + '\n\\end{document}\n'
    out_tex = TEX_DIR / (Path(md_path).stem + '.tex')
    out_tex.write_text(tex, encoding='utf-8')
    return out_tex

## 4. Chạy toàn bộ quy trình

In [10]:
def run_all():
    all_mappings = []
    summary = []
    for pdf_name in PDF_FILES:
        pdf_path = find_pdf_file(pdf_name)
        if pdf_path is None:
            print('Skip missing:', pdf_name)
            continue
        print('\n=== ' + pdf_name + ' ===')
        raw_dir = run_mineru(pdf_path)
        md_path, mappings = normalize_markdown_and_images(pdf_path, raw_dir)
        tex_path = markdown_to_tex(md_path, pdf_path.stem)
        all_mappings.extend(mappings)
        summary.append({'pdf': str(pdf_path), 'markdown': str(md_path), 'tex': str(tex_path), 'images': len(mappings)})
        print('Markdown:', md_path)
        print('TeX:', tex_path)
        print('Images mapped:', len(mappings))
    (FINAL_DIR / 'image_mapping.json').write_text(json.dumps(all_mappings, ensure_ascii=False, indent=2), encoding='utf-8')
    (FINAL_DIR / 'run_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Done:', FINAL_DIR)
    return summary, all_mappings

summary, image_mapping = run_all()


=== Đề 01.pdf ===
Running: mineru -p /content/Đề 01.pdf -o /content/outputs/mineru_raw/de-01 -b pipeline
Start MinerU FastAPI Service: http://127.0.0.1:43147
API documentation: http://127.0.0.1:43147/docs

Markdown: /content/outputs/final/markdown/de-01.md
TeX: /content/outputs/final/tex/de-01.tex
Images mapped: 3

=== Đề 3 HSG.pdf ===
Running: mineru -p /content/Đề 3 HSG.pdf -o /content/outputs/mineru_raw/de-3-hsg -b pipeline
Start MinerU FastAPI Service: http://127.0.0.1:45649
API documentation: http://127.0.0.1:45649/docs

Markdown: /content/outputs/final/markdown/de-3-hsg.md
TeX: /content/outputs/final/tex/de-3-hsg.tex
Images mapped: 6

=== Cau truc de thi HSG 10.pdf ===
Running: mineru -p /content/Cau truc de thi HSG 10.pdf -o /content/outputs/mineru_raw/cau-truc-de-thi-hsg-10 -b pipeline
Start MinerU FastAPI Service: http://127.0.0.1:44851
API documentation: http://127.0.0.1:44851/docs

Markdown: /content/outputs/final/markdown/cau-truc-de-thi-hsg-10.md
TeX: /content/outputs/fin

## 5. Kiểm tra mapping và lấy output

Notebook sẽ tạo `/content/pdf_to_tex_outputs.zip`. Khi chạy bằng Colab extension, lấy file này qua Colab `Contents` view hoặc dùng `Colab: Mount Server To Workspace...`.

In [ ]:
def escape_tex_text(s):
    repl = {
        '&': r'\&', '%': r'\%', '#': r'\#', '_': r'\_', '{': r'\{', '}': r'\}',
        '~': r'\textasciitilde{}', '^': r'\textasciicircum{}',
        '−': '-', '–': '-', '—': '-', '': '<', '': "'", '': r'$^\circ$', '°': r'$^\circ$'
    }
    return ''.join(repl.get(ch, ch) for ch in s)

In [ ]:
import html
from html.parser import HTMLParser

EX_TEST_DIR = FINAL_DIR / 'ex_test'
QUESTION_START_RE2 = re.compile(r'^(?:Câu|Cau)\s*(\d+)\s*[:\.]?\s*(.*)$', re.IGNORECASE)
CHOICE_RE2 = re.compile(r'(?<![\w\\])([A-D])\s*[\.]\s*')
HTML_TABLE_RE2 = re.compile(r'<table.*?</table>', re.IGNORECASE | re.DOTALL)

class SimpleTableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows = []
        self.current_row = None
        self.current_cell = None
    def handle_starttag(self, tag, attrs):
        if tag.lower() == 'tr':
            self.current_row = []
        elif tag.lower() in {'td', 'th'} and self.current_row is not None:
            self.current_cell = []
    def handle_endtag(self, tag):
        if tag.lower() in {'td', 'th'} and self.current_cell is not None:
            self.current_row.append(html.unescape(''.join(self.current_cell)).strip())
            self.current_cell = None
        elif tag.lower() == 'tr' and self.current_row is not None:
            self.rows.append(self.current_row)
            self.current_row = None
    def handle_data(self, data):
        if self.current_cell is not None:
            self.current_cell.append(data)

def html_table_to_tex2(match):
    parser = SimpleTableParser()
    parser.feed(match.group(0))
    rows = [row for row in parser.rows if row]
    if not rows:
        return ''
    width = max(len(row) for row in rows)
    spec = '|' + '|'.join(['c'] * width) + '|'
    out = [r'\begin{center}', r'\begin{tabular}{' + spec + '}', r'\hline']
    for row in rows:
        padded = row + [''] * (width - len(row))
        out.append(' & '.join(inline_md_to_tex(cell) for cell in padded) + r' \\')
        out.append(r'\hline')
    out += [r'\end{tabular}', r'\end{center}']
    return '\n'.join(out)

def is_raw_tex_line2(line):
    stripped = line.strip()
    return stripped.startswith((r'\begin{', r'\end{', r'\hline')) or (' & ' in stripped and stripped.endswith(r'\\'))

def split_questions2(markdown):
    preface = []
    questions = []
    current = None
    text = HTML_TABLE_RE2.sub(html_table_to_tex2, markdown)
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            if current is not None:
                current['lines'].append('')
            continue
        match = QUESTION_START_RE2.match(line)
        if match:
            if current is not None:
                questions.append(current)
            current = {'number': match.group(1), 'lines': [match.group(2).strip()] if match.group(2).strip() else []}
        elif current is not None:
            current['lines'].append(line)
        else:
            preface.append(line)
    if current is not None:
        questions.append(current)
    return preface, questions

def normalize_ex_image_path(src):
    return 'Images/' + Path(str(src)).name

def convert_md_images_to_tex(line):
    return MD_IMG_RE.sub(lambda m: '\n'.join([r'\begin{center}', r'\includegraphics[width=0.82\linewidth]{' + normalize_ex_image_path(m.group(1)) + '}', r'\end{center}']), line)

def split_choices2(line):
    matches = list(CHOICE_RE2.finditer(line))
    if [m.group(1) for m in matches] != ['A', 'B', 'C', 'D']:
        return None
    stem = line[:matches[0].start()].strip()
    choices = []
    for idx, match in enumerate(matches):
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(line)
        choices.append(line[start:end].strip())
    if not all(choices):
        return None
    return stem, choices

def question_to_ex_tex2(question):
    body = [r'\begin{ex}']
    first_text_done = False
    for raw in question['lines']:
        line = convert_md_images_to_tex(raw.strip())
        if not line:
            body.append('')
            continue
        if is_raw_tex_line2(line):
            body.append(line)
            continue
        parsed = split_choices2(line)
        if parsed:
            stem, choices = parsed
            if stem:
                body.append(inline_md_to_tex(stem) + r'\\')
            body.append(r'\choice')
            for choice in choices:
                body.append('{' + inline_md_to_tex(choice) + '}')
            first_text_done = True
            continue
        if line.startswith('#'):
            body.append(r'\subsection*{' + inline_md_to_tex(line.lstrip('#').strip()) + '}')
            continue
        body.append(inline_md_to_tex(line) + (r'\\' if not first_text_done else r'\par'))
        first_text_done = True
    body += [r'\loigiai{}', r'\end{ex}']
    return '\n'.join(body)

def render_ex_test_document2(slug, markdown):
    preface, questions = split_questions2(markdown)
    title = next((line.lstrip('#').strip() for line in preface if line.startswith('#')), slug.replace('-', ' ').title())
    lines = [
        r'\documentclass[12pt,a4paper,oneside]{article}',
        r'\usepackage[top=2cm, bottom=2cm, left=1.5cm, right=1.5cm]{geometry}',
        r'\usepackage{amsmath, amssymb, fancyhdr}',
        r'\usepackage{tkz-euclide,tikz-3dplot,tikz,tkz-tab}',
        r'\usepackage{fontawesome5}',
        r'\usepackage[dethi]{ex_test}',
        r'\usetikzlibrary{shapes.geometric,arrows,decorations.pathmorphing,calc,intersections,angles}',
        r'\usepackage{pgfplots}',
        r'\usepgfplotslibrary{fillbetween}',
        r'\pgfplotsset{compat=1.9}',
        r'\usepackage[hidelinks,unicode]{hyperref}',
        r'\usepackage{esvect}',
        r'\def\vec{\vv}',
        r'\def\overrightarrow{\vv}',
        r'\newcommand{\hoac}[1]{\left[\begin{aligned}#1\end{aligned}\right.}',
        r'\newcommand{\heva}[1]{\left\{\begin{aligned}#1\end{aligned}\right.}',
        r'\renewcommand{\baselinestretch}{1.2}',
        r'\renewtheorem{ex}{\color{red}Câu}',
        r'\begin{document}',
        r'\OPTN{kindTF=t, kindSA=}',
        r'\def\SSS{}',
        r'\Opensolutionfile{ansbook}[ans/ansbook]',
        r'\Opensolutionfile{ans}[ans/ans]',
        r'\OPTN{kindDrag=1}',
        r'\begin{center}\bf\Large',
        inline_md_to_tex(title),
        r'\end{center}',
    ]
    for line in preface:
        if line and not line.startswith('#'):
            lines.append(inline_md_to_tex(line) + r'\par')
    for question in questions:
        lines.append(question_to_ex_tex2(question))
    lines += [r'\Closesolutionfile{ans}', r'\Closesolutionfile{ansbook}', r'\begin{center}\bf\Large', 'BẢNG ĐÁP ÁN', r'\end{center}', r'\inputansbox[0]{7}{ans/ans.tex}', r'\end{document}']
    return '\n'.join(lines) + '\n'

def build_ex_test_outputs():
    EX_TEST_DIR.mkdir(parents=True, exist_ok=True)
    summary = []
    sty_candidates = [Path('/content/ex_test.sty'), Path('/content/ex_test/ex_test.sty'), Path('/content/HSG tinh/ex_test/ex_test.sty')]
    sty_source = next((p for p in sty_candidates if p.exists()), None)
    for md_path in sorted(MD_DIR.glob('*.md')):
        slug = md_path.stem
        target = EX_TEST_DIR / slug
        images_target = target / 'Images'
        ans_target = target / 'ans'
        images_target.mkdir(parents=True, exist_ok=True)
        ans_target.mkdir(parents=True, exist_ok=True)
        if sty_source is not None:
            shutil.copy2(sty_source, target / 'ex_test.sty')
        for image in IMG_DIR.glob(slug + '_*'):
            shutil.copy2(image, images_target / image.name)
        tex = render_ex_test_document2(slug, md_path.read_text(encoding='utf-8', errors='ignore'))
        tex_path = target / (slug + '.tex')
        tex_path.write_text(tex, encoding='utf-8')
        (ans_target / 'ans.tex').write_text('', encoding='utf-8')
        (ans_target / 'ansbook.tex').write_text('', encoding='utf-8')
        summary.append({'slug': slug, 'tex': str(tex_path), 'images': len(list(images_target.glob('*'))), 'has_ex_test_sty': sty_source is not None})
    (EX_TEST_DIR / 'conversion_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Created ex_test outputs:', EX_TEST_DIR)
    return summary

ex_test_summary = build_ex_test_outputs()
ex_test_summary

In [11]:
import pandas as pd, shutil

df = pd.DataFrame(image_mapping)
display(df.head(50))
if not df.empty:
    display(df.groupby(['document', 'question']).size().reset_index(name='image_count'))
    unknown = df[df['question'].astype(str).eq('unknown')]
    print('Unknown mappings:', len(unknown))
    if len(unknown):
        display(unknown)

zip_path = shutil.make_archive('/content/pdf_to_tex_outputs', 'zip', FINAL_DIR)
print('Created:', zip_path)
print('Use Colab Contents view or Colab: Mount Server To Workspace... to copy this zip back to your local workspace.')

,document,document_slug,question,image,source_image,markdown_path
0,Đề 01.pdf,de-01,7,de-01_cau07_01.jpg,/content/outputs/mineru_raw/de-01/Đề 01/auto/i...,../images/de-01_cau07_01.jpg
1,Đề 01.pdf,de-01,9,de-01_cau09_01.jpg,/content/outputs/mineru_raw/de-01/Đề 01/auto/i...,../images/de-01_cau09_01.jpg
2,Đề 01.pdf,de-01,10,de-01_cau10_01.jpg,/content/outputs/mineru_raw/de-01/Đề 01/auto/i...,../images/de-01_cau10_01.jpg
3,Đề 3 HSG.pdf,de-3-hsg,6,de-3-hsg_cau06_01.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau06_01.jpg
4,Đề 3 HSG.pdf,de-3-hsg,7,de-3-hsg_cau07_01.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau07_01.jpg
5,Đề 3 HSG.pdf,de-3-hsg,9,de-3-hsg_cau09_01.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau09_01.jpg
6,Đề 3 HSG.pdf,de-3-hsg,10,de-3-hsg_cau10_01.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau10_01.jpg
7,Đề 3 HSG.pdf,de-3-hsg,10,de-3-hsg_cau10_02.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau10_02.jpg
8,Đề 3 HSG.pdf,de-3-hsg,1,de-3-hsg_cau01_01.jpg,/content/outputs/mineru_raw/de-3-hsg/Đề 3 HSG/...,../images/de-3-hsg_cau01_01.jpg
9,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,10-hsg-tran-bien-2026-azota,7,10-hsg-tran-bien-2026-azota_cau07_01.jpg,/content/outputs/mineru_raw/10-hsg-tran-bien-2...,../images/10-hsg-tran-bien-2026-azota_cau07_01...


,document,question,image_count
0,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,10,2
1,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,12,1
2,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,14,1
3,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,16,1
4,10-HSG-TRẤN BIÊN 2026-AZOTA.pdf,7,1
5,Đề 01.pdf,10,1
6,Đề 01.pdf,7,1
7,Đề 01.pdf,9,1
8,Đề 3 HSG.pdf,1,1
9,Đề 3 HSG.pdf,10,2


Unknown mappings: 0
Created: /content/pdf_to_tex_outputs.zip
Use Colab Contents view or Colab: Mount Server To Workspace... to copy this zip back to your local workspace.


## 6. Tùy chọn biên dịch TeX trên Colab

Cell dưới đây bị comment vì cài TeX Live có thể lâu. Bỏ comment nếu muốn thử xuất PDF từ `.tex` ngay trên Colab.

In [12]:
# sh('apt-get update -qq && apt-get install -y -qq texlive-xetex texlive-lang-other texlive-fonts-recommended')
# for tex_file in TEX_DIR.glob('*.tex'):
#     sh(f'xelatex -interaction=nonstopmode -halt-on-error -output-directory {TEX_DIR} {tex_file}')
